In [1]:
import numpy as np 
import jax 
import jax.numpy as jnp
import matplotlib.pyplot as plt

from projects.trial_to_trial_variability.spec import load_and_process_data

In [2]:
# read the param values from the config yaml 
import yaml
with open('projects/trial_to_trial_variability/config.yaml', 'r') as f:
    config = yaml.safe_load(f)['data_processing_params']

data = load_and_process_data(**config)

In [15]:
X = data[0]
Y = data[1]

X_stim = X['stimulus']
X_train = X['train']
X_test = X['test']

Y_stim = Y['stimulus']
Y_train = Y['train']
Y_test = Y['test']

In [16]:
def neuron_model(theta,
                theta_pref_1=0.0,
                baseline=0.0,
                amplitude_1=1.0,
                width_ccw_1=1.0,
                width_cw_1=1.0,
                exponent_1=2.0,
                theta_pref_2=jnp.pi,
                amplitude_2=0.0,
                width_ccw_2=1.0,
                width_cw_2=1.0,
                exponent_2=2.0):

    min_width = 5e-2
    eps = 1e-12
    min_exponent, max_exponent = 0.1, 5.0
    width_ccw_1, width_cw_1 = jnp.clip(width_ccw_1, min_width, None), jnp.clip(width_cw_1, min_width, None)
    width_ccw_2, width_cw_2 = jnp.clip(width_ccw_2, min_width, None), jnp.clip(width_cw_2, min_width, None)
    exponent_1, exponent_2 = jnp.clip(exponent_1, min_exponent, max_exponent), jnp.clip(exponent_2, min_exponent, max_exponent)
    baseline = jnp.clip(baseline, 0.0, None)
    amplitude_1, amplitude_2 = jnp.clip(amplitude_1, 0.0, None), jnp.clip(amplitude_2, 0.0, None)

    def _signed_circ_diff_rad(angle_radians, preferred_angle_radians):
        delta = angle_radians - preferred_angle_radians
        return jnp.arctan2(jnp.sin(delta), jnp.cos(delta))
        
    signed_diff_1 = _signed_circ_diff_rad(theta, theta_pref_1) + eps  # Add small epsilon to avoid log(0) issues
    width_1_effective = jnp.where(signed_diff_1 < 0, width_ccw_1, width_cw_1)
    width_1_effective = jnp.maximum(width_1_effective, 1e-6)
    peak1_component = amplitude_1 * jnp.exp(-0.5 * (jnp.abs(signed_diff_1) / width_1_effective) ** exponent_1)

    signed_diff_2 = _signed_circ_diff_rad(theta, theta_pref_2) + eps  # Add small epsilon to avoid log(0) issues
    width_2_effective = jnp.where(signed_diff_2 < 0, width_ccw_2, width_cw_2)
    width_2_effective = jnp.maximum(width_2_effective, 1e-6)
    peak2_component = amplitude_2 * jnp.exp(-0.5 * (jnp.abs(signed_diff_2) / width_2_effective) ** exponent_2)
    return baseline + peak1_component + peak2_component

def parameter_estimator(stimuli, spike_counts):
    """
    Estimates parameters for the neuron_model_v3 based on stimulus angles and observed spike counts.
    This estimator employs statistical principles to identify and characterize tuning curve peaks,
    including baseline, amplitude, asymmetric widths, preferred directions, and exponents for
    two potential peaks.

    Parameters are estimated directly from a smoothed firing rate curve, aiming for a simple
    yet robust initial estimation suitable for generalized Gaussian-like tuning profiles.

    Parameters:
    stimuli (np.ndarray): An array of stimulus angles in radians (0 to 2*pi).
    spike_counts (np.ndarray): An array of spike counts corresponding to each stimulus.

    Returns:
    np.ndarray: An array containing the estimated parameters in the following order:
                [theta_pref_1, baseline, amplitude_1, width_ccw_1, width_cw_1,
                exponent_1, theta_pref_2, amplitude_2, width_ccw_2, width_cw_2, exponent_2]
    """
    # --- Configuration Constants ---
    n_bins = 256  # Number of angular bins for tuning curve estimation
    kernel_sigma = 2.5 # Sigma for Gaussian smoothing kernel
    min_peak_amplitude = 0.5 # Min amplitude (spikes/stimulus) above baseline for a peak to be considered valid
    min_model_width = 1e-6 # Minimum allowed width for numerical stability (as per model)
    default_width_value = 1.0 # Default width for non-significant peaks or non-determinable widths
    min_exponent = 0.1 # Minimum allowed exponent value
    max_exponent = 5.0 # Maximum allowed exponent value
    default_exponent_value = 2.0 # Default exponent (Gaussian)
    min_second_peak_ratio = 0.1 # Min amplitude of secondary peak relative to primary
    min_second_peak_separation = np.pi / 4 # Min angular separation between primary and secondary peaks

    # --- 1. Binning and Smoothing ---
    # Convert stimuli to bin indices, handle wrap-around implicitly by modulo in bincount
    bin_idx = ((stimuli * n_bins) / (2 * np.pi)).astype(np.int32)
    bin_idx = np.clip(bin_idx, 0, n_bins - 1)

    sums = np.bincount(bin_idx, weights=spike_counts, minlength=n_bins)
    counts = np.bincount(bin_idx, minlength=n_bins)

    # Create Gaussian smoothing kernel
    kernel_radius = int(3 * kernel_sigma)
    x_kernel = np.arange(-kernel_radius, kernel_radius + 1)
    kernel = np.exp(-0.5 * (x_kernel / kernel_sigma) ** 2)
    kernel /= (np.sum(kernel) + 1e-8) # Normalize kernel

    # Pad arrays for circular convolution
    pad = len(kernel) // 2
    sums_padded = np.pad(sums, (pad, pad), mode='wrap')
    counts_padded = np.pad(counts, (pad, pad), mode='wrap')

    # Convolve to get smoothed sum of spikes and counts
    num_conv = np.convolve(sums_padded, kernel, mode='valid')
    den_conv = np.convolve(counts_padded, kernel, mode='valid')

    # Calculate smoothed tuning curve (avoid division by zero)
    tuning_curve = np.zeros_like(num_conv, dtype=float)
    valid_den_mask = den_conv > 1e-8
    tuning_curve[valid_den_mask] = num_conv[valid_den_mask] / den_conv[valid_den_mask]

    angle_step = 2 * np.pi / n_bins

    # --- 2. Baseline Estimation ---
    # Baseline cannot be negative
    baseline_est = np.maximum(0.0, np.min(tuning_curve))

    # --- Helper function for estimating peak shape parameters ---
    def _get_peak_params_simple(peak_idx_val, peak_idx, bsl, tc, n_bns, ang_step,
                                min_w, def_w, min_exp, max_exp, def_exp, min_amp_thresh):
        amp = peak_idx_val - bsl
        if amp < min_amp_thresh:
            return amp, def_w, def_w, def_exp # Return default parameters if amplitude too small

        # Find bins where tuning curve drops below half-max
        target_half_val = bsl + amp / 2.0
        half_ccw_bins, half_cw_bins = 0, 0
        for k in range(1, n_bns // 2 + 1):
            if half_ccw_bins == 0 and tc[(peak_idx - k + n_bns) % n_bns] <= target_half_val:
                half_ccw_bins = k
            if half_cw_bins == 0 and tc[(peak_idx + k) % n_bns] <= target_half_val:
                half_cw_bins = k
            if half_ccw_bins > 0 and half_cw_bins > 0:
                break
        
        # Calculate width (v1-style, implicitly assuming Gaussian form for width parameter base)
        SQRT_2_LOG_2 = np.sqrt(2 * np.log(2)) 
        width_ccw = (half_ccw_bins * ang_step) / SQRT_2_LOG_2 if half_ccw_bins > 0 else def_w
        width_cw = (half_cw_bins * ang_step) / SQRT_2_LOG_2 if half_cw_bins > 0 else def_w
        
        # Clip widths to valid range
        width_ccw = np.clip(width_ccw, min_w, np.pi)
        width_cw = np.clip(width_cw, min_w, np.pi)

        # Find bins where tuning curve drops below quarter-max for exponent estimation
        target_qtr_val = bsl + amp / 4.0
        qtr_ccw_bins, qtr_cw_bins = 0, 0
        for k in range(1, n_bns // 2 + 1):
            if qtr_ccw_bins == 0 and tc[(peak_idx - k + n_bns) % n_bns] <= target_qtr_val:
                qtr_ccw_bins = k
            if qtr_cw_bins == 0 and tc[(peak_idx + k) % n_bns] <= target_qtr_val:
                qtr_cw_bins = k
            if qtr_ccw_bins > 0 and qtr_cw_bins > 0:
                break
        
        # Exponent estimation (log(2) / log(dist_qtr / dist_half))
        exponent_estimates = []
        if half_ccw_bins > 0 and qtr_ccw_bins > half_ccw_bins: # Ensure ratio > 1 for valid log
            exponent_estimates.append(np.log(2) / np.log(qtr_ccw_bins / half_ccw_bins))
        if half_cw_bins > 0 and qtr_cw_bins > half_cw_bins:
            exponent_estimates.append(np.log(2) / np.log(qtr_cw_bins / half_cw_bins))
        
        # Take mean of valid estimates, else default
        exponent = np.mean(exponent_estimates) if exponent_estimates else def_exp
        exponent = np.clip(exponent, min_exp, max_exp)
        
        return amp, width_ccw, width_cw, exponent

    # --- 3. Peak Identification and Parameter Estimation ---
    # Find all local maxima in the smoothed tuning curve
    local_maxima = []
    for i in range(n_bins):
        prev_val = tuning_curve[(i - 1 + n_bins) % n_bins]
        next_val = tuning_curve[(i + 1) % n_bins]
        if tuning_curve[i] >= prev_val and tuning_curve[i] >= next_val:
            local_maxima.append((tuning_curve[i], i))
    local_maxima.sort(key=lambda x: x[0], reverse=True) # Sort by peak amplitude (descending)

    # Initialize all output parameters with model defaults (if no peak is found)
    theta_pref_1 = 0.0
    amplitude_1 = default_width_value # amplitude for primary peak default to 1 for non-zero contribution if peak is small
    width_ccw_1, width_cw_1, exponent_1 = default_width_value, default_width_value, default_exponent_value
    
    theta_pref_2 = np.pi # Antipodal to default theta_pref_1
    amplitude_2 = 0.0 # Default no second peak
    width_ccw_2, width_cw_2, exponent_2 = default_width_value, default_width_value, default_exponent_value

    # Process Primary Peak
    if local_maxima:
        peak_1_val, peak_1_idx = local_maxima[0]
        # Estimate parameters for the primary (highest) peak
        amplitude_1, width_ccw_1, width_cw_1, exponent_1 = \
            _get_peak_params_simple(peak_1_val, peak_1_idx, baseline_est, tuning_curve, n_bins, angle_step,
                                    min_model_width, default_width_value, min_exponent, max_exponent,
                                    default_exponent_value, min_peak_amplitude)
        theta_pref_1 = peak_1_idx * angle_step
        
        # Process Secondary Peak (if a suitable candidate exists)
        if len(local_maxima) > 1:
            # Iterate through other local maxima to find a suitable secondary peak
            for i in range(1, len(local_maxima)):
                peak_2_val_candidate, peak_2_idx_candidate = local_maxima[i]
                current_amplitude_2_candidate = peak_2_val_candidate - baseline_est
                current_theta_pref_2_candidate = peak_2_idx_candidate * angle_step
                
                # Check amplitude significance relative to primary peak's estimated amplitude
                if current_amplitude_2_candidate < (amplitude_1 * min_second_peak_ratio):
                    continue # Skip if too small compared to the primary peak
                
                # Check angular separation from the *primary* peak
                # np.arctan2(sin(a-b), cos(a-b)) gives circular distance in [-pi, pi]
                peak_sep_rad = np.abs(np.arctan2(np.sin(theta_pref_1 - current_theta_pref_2_candidate),
                                                np.cos(theta_pref_1 - current_theta_pref_2_candidate)))
                if peak_sep_rad < min_second_peak_separation:
                    continue # Skip if too close to the primary peak

                # Valid secondary peak found, estimate its parameters
                amplitude_2, width_ccw_2, width_cw_2, exponent_2 = \
                    _get_peak_params_simple(peak_2_val_candidate, peak_2_idx_candidate, baseline_est, tuning_curve, n_bins, angle_step,
                                            min_model_width, default_width_value, min_exponent, max_exponent,
                                            default_exponent_value, min_peak_amplitude)
                theta_pref_2 = current_theta_pref_2_candidate
                
                # Break after finding the first suitable secondary peak (highest amplitude among remaining candidates)
                break
    
    # --- Pack and return parameters in the correct order for neuron_model_v3 ---
    return np.array([theta_pref_1, baseline_est, amplitude_1, width_ccw_1, width_cw_1,
                    exponent_1, theta_pref_2, amplitude_2, width_ccw_2, width_cw_2, exponent_2])

# Per-sample loss function
def loss_single_sample(params, x_i, y_i):
    y_pred = neuron_model(x_i, params)
    if y_i.ndim == 1:
        y_i = y_i[None, :]
    if y_pred.ndim == 1:
        y_pred = y_pred[None, :]
    sample_loss = jnp.asarray(loss_fn(y_pred, y_i))
    if sample_loss.ndim == 0:
        return sample_loss
    return jnp.mean(sample_loss)

def _optimize_params(flat_params, x, y, learning_rate=1e-3, max_iter=500):
    learning_rate_local = float(learning_rate)
    opt = optax.adam(learning_rate_local, b1=0.9, b2=0.999, eps=1e-8)
    opt_state = opt.init(flat_params)

    # Vectorize over samples
    # params: pytree (batched), x: (n_samples, n_features, n_trials_x), y: (n_samples, n_targets, n_trials_y)
    # Output: (n_samples,)
    loss_total = jax.vmap(loss_single_sample, in_axes=(0, 0, 0), out_axes=0)

    loss_param = lambda params: jnp.mean(loss_total(unflatten(params), x, y))
    loss_param_and_grad = jax.value_and_grad(loss_param)

    @jax.jit
    def train_step(params, opt_state):
        loss, grad = loss_param_and_grad(params)
        updates, opt_state = opt.update(grad, opt_state, params)
        params = optax.apply_updates(params, updates)
        return params, opt_state, loss

    print_every = 20
    params = flat_params
    initial_loss = loss_param(params)
    best_loss, best_params = initial_loss.copy(), params.copy()
    for step in range(1, max_iter + 1):
        _check_timeout()
        params, opt_state, loss_val = train_step(params, opt_state)
        _check_timeout()
        if jnp.isnan(loss_val) or jnp.isinf(loss_val) or jnp.any(jnp.isnan(params)) or jnp.any(jnp.isinf(params)):
            logging.info(f"Loss is NaN or Inf at step {step}. Stopping optimization.")
            print(f"Final loss: {loss_val:.4f} at step {step}")
            break
        if loss_val < best_loss:
            best_loss = loss_val.copy()
            best_params = params.copy()
        if step % print_every == 0:
            print(f"step {step:4d}  loss {loss_val:.4f}")
    params = unflatten(best_params)
    print(f"params optimized. Loss: {best_loss:.4f}")
    return params

stimuli = jnp.array(X_stim) # shape (n_source, n_time)
source_train = jnp.array(X_train) # shape (n_source, n_time)
target_train = jnp.array(Y_train) # shape (n_target, n_time)

# Calculate the initial parameters for all source cells 
source_params_init = jax.vmap(parameter_estimator)(stimuli, source_train) # shape (n_source, n_params)
# use gradient descent to optimise the parameters 
source_params = _optimize_params(source_params_init, stimuli, source_train)

target_params_init = jax.vmap(parameter_estimator)(stimuli, target_train) # shape (n_target, n_params)
target_params = _optimize_params(target_params_init, stimuli, target_train)

def model_v1(X, params):
    """ Gain Modulation + per cell modulation

    Equation : For each target cell c at timepoint t with stimulus angle theta, 
        f(theta, t; cell_params) = a(t) * g(theta(t) ; cell_params) + b(t) * c
    where g(theta(t); cell_params) is some tuning function. 

    Args : 
        X (dict) : Inputs object with keys 'stimulus', 'train', 'test'. Each with shape (n_source_cells, n_time)
        params (dict) : Parameter dictionary with keys: 
            - gain_modulation : a(t) (shape (n_time,))
            - baseline_modulation : b(t) (shape (n_time,))
            - source_tuning_params : parameters for the tuning function g(theta) (shape (n_source_cells, n_params))
            - target_tuning_params : parameters for the tuning function g(theta) (shape (n_target_cells, n_params))
            - target_cell_modulation : c (shape (n_target_cells,)) # this parameter will be fit using gradient descent

    Returns : 
        jnp.ndarray : Predicted responses for the target cells with shape (n_target_cells, n_time)
    """
    stimuli = jnp.array(X['stimulus']) # shape (n_source, n_time)
    source_train = jnp.array(X['train']) # shape (n_source, n_time)

    # Step 1 : For every source cell, fit a peaky tuning curve 
    # Step 2 : Use ALS (Alternating Least Squares) to fit a(t), b(t), and g(theta) iteratively.

    source_tuning_params = params['source_tuning_params'] # shape (n_source_cells, n_params)
    target_tuning_params = params['target_tuning_params'] # shape (n_target_cells, n_params)
    
    # Calculate the predicted response for each source cell based on the optimised parameters and stimulus 
    g_source = jax.vmap(neuron_model, in_axes=(0, 0))(stimuli, source_tuning_params).T # shape (n_time, n_source)  

    n_t, n_source = g_source.shape

    # Use ALS to fit a(t) and b(t) given g_source and the target responses.
    a = jnp.ones(n_t) # shape (n_time,)
    b = jnp.zeros(n_t) # shape (n_time,)
    prev_err = np.inf

    # initial guess for per cell modulation
    c = jnp.ones(n_source) # shape (n_source,)

    max_iter = 100
    tol = 1e-2
    Y = source_train.T # shape (n_time, n_source)
    for i in range(max_iter):
        # -- Step A : Solve for a(t) and b(t) -- (fix c)
        for t in range(n_t):
            X_linalg = jnp.stack([g_source[t, :], c], axis=1) # shape (n_source, 2)
            # Solve Y_t = X * [a_t, b_t].T 
            coefs, _, _, _ = jnp.linalg.lstsq(X_linalg, Y[t, :], rcond=None)
            a[t], b[t] = coeffs[0], coeffs[1]

        # Step B ; Solve for c (fix a(t) and b(t))
        residual = Y - (a * g_source)
        c = (residual @ b) / (b @ b)

        # Step C : Normalisation 
        # Normalise c to unit length to keep the scale stable
        norm_val = jnp.linalg.norm(c)
        c /= norm_val
        b *= norm_val

        if abs(prev_err - err) < tol:
            print(f"ALS converged at iteration {i}. Error: {err:.4f}")
            break
        prev_err = err
    
    # calculate c of target cells using the optimised a(t) and b(t) and the target responses
    g_target = jax.vmap(neuron_model, in_axes=(0, 0))(stimuli, target_tuning_params).T # shape (n_time, n_target)
    return a[:, None] * g_target + b[:, None] * c[None, :]    

# So we need to first first fit a(t), b(t), and g from the source cells. 
# Then the param 

TracerArrayConversionError: The numpy.ndarray conversion method __array__() was called on traced array with shape int32[4282]
This BatchTracer with object id 135883614601968 was created on line:
  /tmp/ipykernel_1507393/1678483960.py:71:14 (parameter_estimator)
See https://docs.jax.dev/en/latest/errors.html#jax.errors.TracerArrayConversionError